Section 1: Mount Google Drive

In [ ]:
# Import the drive module from google.colab
from google.colab import drive

# Mount Google Drive to access files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Section 2: Extract Dataset

In [ ]:
import zipfile
import os

# Define the path to your zip file in Google Drive
zip_path = '/content/drive/MyDrive/Data Science/bus-stop-assess final.v2i.yolov8.zip'
# Define where to extract the dataset
extract_path = '/content/dataset'

# Create the directory if it doesn't exist
if not os.path.exists(extract_path):
    os.makedirs(extract_path)

# Unzip the dataset into the local Colab directory
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraction from Drive completed!")

Dataset extraction from Drive completed!


Section 3: Configure data.yaml

In [ ]:
import yaml

# Path to the dataset configuration file
yaml_path = '/content/dataset/data.yaml'

# Load the existing yaml data
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update paths to match the local Colab environment
data['train'] = '/content/dataset/train/images'
data['val'] = '/content/dataset/valid/images'
data['test'] = '/content/dataset/test/images'

# Save the updated configuration back to the file
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("Configuration file updated successfully!")


Configuration file updated successfully!


Section 4: Training the Model (Weights & Training)

In [ ]:
!pip install ultralytics
import os
from ultralytics import YOLO

# 1. Load the pre-trained weights (YOLOv5)
model = YOLO('yolov5nu.pt')

# 2. Start the training process
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    project='/content/drive/MyDrive/Thesis_Project_YOLOv5/training_results',
    name='bus_stop_v5_baseline',
    scale=0.5,
    perspective=0.0005,
    mosaic=1.0,
    mixup=0.2,
    degrees=10.0
)
print("YOLOv5 Training completed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epoc

Section 5: Model Inference (Testing on New Images)

After training, we use the best-trained weights (best.pt) to run predictions on the test dataset to visualize how the model performs on unseen data.

In [ ]:
import os
from ultralytics import YOLO

model_path = '/content/drive/MyDrive/Thesis_Project_YOLOv5/training_results/bus_stop_v5_baseline/weights/best.pt'
model = YOLO(model_path)

test_images_path = '/content/dataset/test/images'

if os.path.exists(test_images_path):
    results = model.predict(
        source=test_images_path,
        save=True,
        conf=0.25,
        project='/content/drive/MyDrive/Thesis_Project_YOLOv5',
        name='test_results_v5'
    )
    print("Inference completed for YOLOv5!")
else:
    print("Error: Test images directory not found.")


image 1/118 /content/dataset/test/images/1F5kE4ER0UZz8BDchBzLcA_jpg.rf.0c300b9c20a106c0a3a8f41cfb34a295.jpg: 640x640 1 seating, 1 shelter, 1 trash can, 7.8ms
image 2/118 /content/dataset/test/images/210170_1_jpg.rf.6f4e4fedb92493635b2ff2f16010db06.jpg: 640x640 1 sign, 10.6ms
image 3/118 /content/dataset/test/images/211401_0_jpg.rf.a4671744c3127b679eede33211c5d91b.jpg: 640x640 1 seating, 1 trash can, 7.1ms
image 4/118 /content/dataset/test/images/211997_0_jpg.rf.2bb3ca346f3f71378b912889146b78d9.jpg: 640x640 1 seating, 1 sign, 1 trash can, 7.4ms
image 5/118 /content/dataset/test/images/212079_0_jpg.rf.4f75a99bfddaa3e3f2a0eda98319c7ec.jpg: 640x640 1 route info, 1 seating, 1 sign, 1 trash can, 7.4ms
image 6/118 /content/dataset/test/images/212106_1_jpg.rf.b7db26c24c34b7e3e3d9fa99edb912dd.jpg: 640x640 2 signs, 7.2ms
image 7/118 /content/dataset/test/images/212144_1_jpg.rf.32a2e1d323f5808ee9b4c72e4ee37af1.jpg: 640x640 1 seating, 1 shelter, 1 sign, 2 trash cans, 7.0ms
image 8/118 /content/da

Section 6: Quantitative Evaluation (Final Metrics)

This section calculates the final performance metrics (mAP, Precision, Recall) using the test split of your dataset for the thesis report.

In [ ]:
from ultralytics import YOLO

# 1. Load the best-trained model of YOLOv5
model_path = '/content/drive/MyDrive/Thesis_Project_YOLOv5/training_results/bus_stop_v5_baseline/weights/best.pt'
model = YOLO(model_path)

# 2. Run validation on the test split
metrics = model.val(data='/content/dataset/data.yaml', split='test')

# 3. Print the matrix for the thesis report table
print("-" * 30)
print(f"Final Results for YOLOv5 Baseline:")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")
print("-" * 30)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv5n summary (fused): 85 layers, 2,504,114 parameters, 0 gradients, 7.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2218.9±561.1 MB/s, size: 81.2 KB)
val: Scanning /content/dataset/test/labels... 118 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 2.2Kit/s 0.1s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.4it/s 3.4s
                   all        118        380      0.939      0.766      0.852      0.584
            route info         30         30      0.929      0.567      0.685      0.388
              schedule         33         33      0.931      0.823      0.932      0.699
               seating         71         72      0.928      0.806      0.859      0.545
               shelter         61         61      0.947      0.951      0.966      0.77